In [21]:
# =========================================================
# TF-IDF & VSM
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from src.tfidf.tfidf_process import build_tfidf

In [22]:
# =========================================================
# LOAD DATA HASIL PREPROCESSING
# =========================================================
base_dir = os.path.abspath('..')
csv_path = os.path.join(base_dir, 'data', 'cleaned_papers.csv')

df = pd.read_csv(csv_path)
df['cleaned_text'] = df['cleaned_text'].fillna('')

print(f'✅ Data loaded: {len(df)} baris')
df.head(3)

✅ Data loaded: 200 baris


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,cleaned_text
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,https://pmc.ncbi.nlm.nih.gov/articles/PMC11980...,https://www.nature.com/articles/s41579-023-009...,pdf_failed,machine learning for microbiologists how to ev...
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning,https://par.nsf.gov/servlets/purl/10418406,https://par.nsf.gov/servlets/purl/10418406,pdf_failed,international conference on machine learning i...
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,NaN,https://onlinelibrary.wiley.com/doi/abs/10.100...,no_pdf,what is machine learning that one can employ i...


In [23]:
# =========================================================
# FIT TF-IDF (pakai src/tfidf/tfidf_process.py)
# =========================================================
import json


save_dir = os.path.join(base_dir, 'data', 'tfidf')

print('⚙️ Menghitung TF-IDF...')
vectorizer, tfidf_matrix = build_tfidf(df['cleaned_text'].tolist(), save_dir)

print(f'✅ TF-IDF selesai!')
print(f'📊 Jumlah dokumen : {tfidf_matrix.shape[0]}')
print(f'📊 Jumlah term    : {tfidf_matrix.shape[1]}')

# ambil parameter vectorizer
params = vectorizer.get_params()
def make_json_safe(value):
    if isinstance(value, type):
        return value.__name__
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {k: make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    return value

safe_params = make_json_safe(params)

with open(os.path.join(save_dir, 'vectorizer_params.json'), 'w', encoding='utf-8') as f:
    json.dump(safe_params, f, indent=2, ensure_ascii=False)

⚙️ Menghitung TF-IDF...
✅ TF-IDF selesai!
📊 Jumlah dokumen : 200
📊 Jumlah term    : 1718


In [24]:
# =========================================================
# REPRESENTASI VSM - TOP TERM PER DOKUMEN
# =========================================================
feature_names = vectorizer.get_feature_names_out()

print('\n=== Contoh Representasi VSM ===')
for i in range(min(3, len(df))):
    row = tfidf_matrix[i].toarray()[0]
    top_indices = row.argsort()[::-1][:10]
    top_terms = [(feature_names[j], round(row[j], 4)) for j in top_indices if row[j] > 0]
    print(f'\nDokumen {i+1}: {df["title"].iloc[i][:60]}')
    print(f'Top terms: {top_terms}')



=== Contoh Representasi VSM ===

Dokumen 1: Machine learning for microbiologists
Top terms: [('microbiologists', 0.4736), ('machine', 0.3501), ('learning', 0.3241), ('work', 0.3084), ('how', 0.2933), ('grasp', 0.2368), ('evaluate', 0.2107), ('presented', 0.2107), ('topics', 0.2107), ('allow', 0.2107)]

Dokumen 2: International conference on machine learning
Top terms: [('conference', 0.2429), ('general', 0.2429), ('delineation', 0.2429), ('hierarchical', 0.2429), ('box', 0.2429), ('resolution', 0.2429), ('black', 0.2429), ('roles', 0.2429), ('uncertainty', 0.2429), ('bandits', 0.2429)]

Dokumen 3: What is machine learning?
Top terms: [('learning', 0.4953), ('or', 0.2988), ('either', 0.2412), ('required', 0.2412), ('skills', 0.2412), ('automated', 0.2147), ('employ', 0.2147), ('types', 0.2147), ('unsupervised', 0.1991), ('via', 0.1881)]


In [25]:
# =========================================================
# VERIFIKASI MANUAL TF-IDF
# TF = f_t,d / jumlah kata dokumen
# IDF = log(N / df_t)
# TF-IDF = TF * IDF
# =========================================================
import numpy as np

doc_idx = 0
tokens = str(df['cleaned_text'].iloc[doc_idx]).split()

if len(tokens) == 0:
    print("Dokumen kosong setelah preprocessing.")
else:
    term = tokens[0]  # contoh term
    N = len(df)

    f_td = tokens.count(term)
    tf = f_td / len(tokens)

    df_t = sum(1 for text in df['cleaned_text'].fillna('') if term in str(text).split())
    idf = np.log(N / df_t) if df_t > 0 else 0.0

    tfidf = tf * idf

    print("=== Verifikasi Manual (Sesuai Proposal) ===")
    print(f"Dokumen ke         : {doc_idx}")
    print(f"Term               : '{term}'")
    print(f"f_t,d              : {f_td}")
    print(f"Jumlah kata dokumen: {len(tokens)}")
    print(f"TF                 : {f_td}/{len(tokens)} = {tf:.6f}")
    print(f"N                  : {N}")
    print(f"df_t               : {df_t}")
    print(f"IDF                : log({N}/{df_t}) = {idf:.6f}")
    print(f"TF-IDF             : {tf:.6f} * {idf:.6f} = {tfidf:.6f}")

=== Verifikasi Manual (Sesuai Proposal) ===
Dokumen ke         : 0
Term               : 'machine'
f_t,d              : 4
Jumlah kata dokumen: 32
TF                 : 4/32 = 0.125000
N                  : 200
df_t               : 53
IDF                : log(200/53) = 1.328025
TF-IDF             : 0.125000 * 1.328025 = 0.166003


In [26]:
# =========================================================
# SIMPAN INDEX DOKUMEN
# =========================================================
index_path = os.path.join(base_dir, 'data', 'tfidf', 'doc_index.csv')
df[['id', 'title']].to_csv(index_path, index=False)

print(f'✅ Tersimpan: {index_path}')
print(f'📊 Total: {len(df)} baris')
df[['id', 'title']].head(5)

✅ Tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\doc_index.csv
📊 Total: 200 baris


,id,title
0,1,Machine learning for microbiologists
1,2,International conference on machine learning
2,3,What is machine learning?
3,4,Amnesiac machine learning
4,5,Designing nanotheranostics with machine learning


In [27]:
# =========================================================
# SIMPAN SEMUA BOBOT TF-IDF KE CSV (SPARSE-EFFICIENT)
# =========================================================
feature_names = vectorizer.get_feature_names_out()
coo = tfidf_matrix.tocoo()

tfidf_df = pd.DataFrame({
    'doc_row': coo.row,
    'term_idx': coo.col,
    'tfidf_score': np.round(coo.data, 6)
})

tfidf_df['doc_id'] = df.iloc[tfidf_df['doc_row'].values]['id'].values
tfidf_df['title'] = df.iloc[tfidf_df['doc_row'].values]['title'].values
tfidf_df['term'] = feature_names[tfidf_df['term_idx'].values]

tfidf_df = tfidf_df[['doc_id', 'title', 'term', 'tfidf_score']].sort_values(
    by=['doc_id', 'tfidf_score'],
    ascending=[True, False]
)

save_path = os.path.join(base_dir, 'data', 'tfidf', 'tfidf_weights.csv')
tfidf_df.to_csv(save_path, index=False)

print(f'✅ TF-IDF berhasil disimpan!')
print(f'📁 Lokasi: {save_path}')
print(f'📊 Total baris non-zero: {len(tfidf_df)}')
tfidf_df.head(20)

✅ TF-IDF berhasil disimpan!
📁 Lokasi: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\tfidf_weights.csv
📊 Total baris non-zero: 5308


,doc_id,title,term,tfidf_score
3,1,Machine learning for microbiologists,microbiologists,0.473610
0,1,Machine learning for microbiologists,machine,0.350117
1,1,Machine learning for microbiologists,learning,0.324123
10,1,Machine learning for microbiologists,work,0.308387
4,1,Machine learning for microbiologists,how,0.293297
18,1,Machine learning for microbiologists,grasp,0.236805
6,1,Machine learning for microbiologists,evaluate,0.210744
12,1,Machine learning for microbiologists,topics,0.210744
13,1,Machine learning for microbiologists,presented,0.210744
17,1,Machine learning for microbiologists,allow,0.210744


In [28]:
# =========================================================
# CEK KONSISTENSI QUERY -> VECTOR
# =========================================================
from src.preprocessing.clean_text import clean_text

from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming

stop_words = get_stopwords()

def preprocess_query(query: str) -> str:
    text = clean_text(query)
    tokens = tokenizing(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    if all(token.isascii() for token in tokens):
        return ' '.join(tokens)
    return ' '.join(stemming(tokens))

query = "web development"
q_clean = preprocess_query(query)
q_vec = vectorizer.transform([q_clean])

print(f'Query asli   : {query}')
print(f'Query proses : {q_clean}')
print(f'Non-zero term query vector: {q_vec.nnz}')

Query asli   : web development
Query proses : web development
Non-zero term query vector: 2
